In [1]:
import numpy as np
import pandas as pd

from sklearn.dummy import DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("df_final.csv")

print("Shape:", df.shape)
display(df.head())
print(df.dtypes)

Shape: (56013, 43)


,arrondissement,DATE,green_count,green_total_area,green_calc_area,green_hort_area,green_perimeter,main_green_category,green_area_share_pct,arr_area_m2,...,roads_total_surface_m2_avenue,roads_total_surface_m2_boulevard,roads_total_surface_m2_rue,water_count,water_total_area,main_water_class,dni_wm2,wind_speed_10m_ms,wind_dir_10m_deg,pressure_pa
0,1,1990-05-01,16,50490.0,6731.0,19371.0,947.038315,Jardiniere,0.73,1830000,...,39280.0,56086.00,430080.95,3.0,166300.718992,Emprise voie d'eau : bief,183.455833,2.215833,323.208333,101623.75
1,3,1990-05-01,41,28612.0,28477.0,15489.0,2767.084225,Murs vegetalises,0.41,1170000,...,NaN,64144.50,283466.39,1.0,75.941599,Emprise voie d'eau : bief,183.455833,2.215833,323.208333,101623.75
2,4,1990-05-01,63,72445.0,69480.0,34099.0,10273.509163,Jardiniere,1.04,1600000,...,NaN,48088.75,186609.92,13.0,239021.597553,Emprise voie d'eau : embranchement,183.455833,2.215833,323.208333,101623.75
3,5,1990-05-01,96,93057.0,72374.0,37881.0,10211.666111,Jardiniere,1.34,2540000,...,97327.0,271085.00,441456.80,2.0,433.794234,Emprise voie d'eau : embranchement,183.455833,2.215833,323.208333,101623.75
4,6,1990-05-01,41,31810.0,31389.0,12077.0,3412.889510,Jardiniere,0.46,2150000,...,538.2,134748.00,446949.78,NaN,NaN,NaN,183.455833,2.215833,323.208333,101623.75


arrondissement                        int64
DATE                                    str
green_count                           int64
green_total_area                    float64
green_calc_area                     float64
green_hort_area                     float64
green_perimeter                     float64
main_green_category                     str
green_area_share_pct                float64
arr_area_m2                           int64
green_area_pct                      float64
LAT                                 float64
LON                                 float64
ALTI                                float64
precipitation_mm_24h                float64
temp_min_c                          float64
hour_temp_min                       float64
temp_max_c                          float64
hour_temp_max                       float64
temp_mean_c                         float64
temp_mean_min_max_c                 float64
temp_amplitude_c                    float64
frost_duration_min              

In [2]:
df["DATE"] = pd.to_datetime(df["DATE"], errors="coerce")
df["arrondissement"] = pd.to_numeric(df["arrondissement"], errors="coerce")

print("Date range:", df["DATE"].min(), "to", df["DATE"].max())
print("Number of dates:", df["DATE"].nunique())
print("Arrondissements:", sorted(df["arrondissement"].dropna().unique()))

duplicate_count = df.duplicated(["arrondissement", "DATE"]).sum()
print("Duplicate arrondissement-date rows:", duplicate_count)

if duplicate_count:
    raise ValueError(
        "Resolve duplicate arrondissement-date rows before modeling; "
        "each arrondissement should have at most one row per date."
    )

display(df.isna().mean().sort_values(ascending=False).head(15))

Date range: 1990-05-01 00:00:00 to 2026-09-30 00:00:00
Number of dates: 5661
Arrondissements: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20)]
Duplicate arrondissement-date rows: 0


wind_mean_ms                     0.661829
frost_duration_min               0.484191
hour_temp_max                    0.443004
hour_temp_min                    0.442844
temp_mean_c                      0.440558
water_count                      0.346919
main_water_class                 0.346919
water_total_area                 0.346919
roads_avg_width_m_avenue         0.096942
roads_total_surface_m2_avenue    0.096942
roads_total_length_m_avenue      0.096942
precipitation_mm_24h             0.077143
green_area_pct                   0.000000
green_area_share_pct             0.000000
arr_area_m2                      0.000000
dtype: float64

In [3]:
TARGET_COL = "temp_max_c"  # Or "temp_mean_c" for daily mean temperature.

if TARGET_COL not in df.columns:
    raise ValueError(f"Target column {TARGET_COL!r} is not in the dataset.")

df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")

# Numeric weather measurements that may be useful from the previous day.
# Hour columns are times of occurrence, not temperature measurements, so they
# are deliberately excluded.
weather_cols = [
    "precipitation_mm_24h",
    "temp_min_c",
    "temp_max_c",
    "temp_mean_c",
    "temp_mean_min_max_c",
    "temp_amplitude_c",
    "frost_duration_min",
    "wind_mean_ms",
    "dni_wm2",
    "wind_speed_10m_ms",
    "wind_dir_10m_deg",
    "pressure_pa",
]
weather_cols = [col for col in weather_cols if col in df.columns]

for col in weather_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.sort_values(["arrondissement", "DATE"]).copy()

# Create prior-day weather features. If the previous row is not exactly one day
# earlier, its values are discarded rather than treated as yesterday's weather.
day_gap = df.groupby("arrondissement")["DATE"].diff().dt.days

for col in weather_cols:
    previous_day_values = df.groupby("arrondissement")[col].shift(1)
    df[f"{col}_lag1"] = previous_day_values.where(day_gap.eq(1))

lag_weather_cols = [f"{col}_lag1" for col in weather_cols]

# Numeric morphology and location features.
morphology_candidates = [
    "green_count",
    "green_total_area",
    "green_calc_area",
    "green_hort_area",
    "green_perimeter",
    "green_area_share_pct",
    "arr_area_m2",
    "green_area_pct",
    "LAT",
    "LON",
    "ALTI",
    "roads_total_length_m_avenue",
    "roads_total_length_m_boulevard",
    "roads_total_length_m_rue",
    "roads_avg_width_m_avenue",
    "roads_avg_width_m_boulevard",
    "roads_avg_width_m_rue",
    "roads_total_surface_m2_avenue",
    "roads_total_surface_m2_boulevard",
    "roads_total_surface_m2_rue",
    "water_count",
    "water_total_area",
]
morphology_cols = [col for col in morphology_candidates if col in df.columns]

# Encode annual seasonality using the date, without using same-day weather.
day_of_year = df["DATE"].dt.dayofyear
df["day_of_year_sin"] = np.sin(2 * np.pi * day_of_year / 365.25)
df["day_of_year_cos"] = np.cos(2 * np.pi * day_of_year / 365.25)

feature_cols = morphology_cols + lag_weather_cols + [
    "day_of_year_sin",
    "day_of_year_cos",
]

for col in feature_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("Target:", TARGET_COL)
print("Features:", feature_cols)
print("Number of features:", len(feature_cols))

Target: temp_max_c
Features: ['green_count', 'green_total_area', 'green_calc_area', 'green_hort_area', 'green_perimeter', 'green_area_share_pct', 'arr_area_m2', 'green_area_pct', 'LAT', 'LON', 'ALTI', 'roads_total_length_m_avenue', 'roads_total_length_m_boulevard', 'roads_total_length_m_rue', 'roads_avg_width_m_avenue', 'roads_avg_width_m_boulevard', 'roads_avg_width_m_rue', 'roads_total_surface_m2_avenue', 'roads_total_surface_m2_boulevard', 'roads_total_surface_m2_rue', 'water_count', 'water_total_area', 'precipitation_mm_24h_lag1', 'temp_min_c_lag1', 'temp_max_c_lag1', 'temp_mean_c_lag1', 'temp_mean_min_max_c_lag1', 'temp_amplitude_c_lag1', 'frost_duration_min_lag1', 'wind_mean_ms_lag1', 'dni_wm2_lag1', 'wind_speed_10m_ms_lag1', 'wind_dir_10m_deg_lag1', 'pressure_pa_lag1', 'day_of_year_sin', 'day_of_year_cos']
Number of features: 36


In [4]:
model_df = df.dropna(subset=["DATE", "arrondissement", TARGET_COL]).copy()

unique_dates = np.sort(model_df["DATE"].unique())
if len(unique_dates) < 10:
    raise ValueError("There are too few unique dates for this chronological split.")

i_train_end = int(len(unique_dates) * 0.70)
i_valid_end = int(len(unique_dates) * 0.85)

train_end_date = pd.Timestamp(unique_dates[i_train_end - 1])
valid_end_date = pd.Timestamp(unique_dates[i_valid_end - 1])

train = model_df[model_df["DATE"] <= train_end_date].copy()
valid = model_df[
    (model_df["DATE"] > train_end_date) &
    (model_df["DATE"] <= valid_end_date)
].copy()
test = model_df[model_df["DATE"] > valid_end_date].copy()

for name, split in [("train", train), ("validation", valid), ("test", test)]:
    print(
        f"{name}: {split['DATE'].min().date()} to "
        f"{split['DATE'].max().date()} | rows={len(split)} | "
        f"target rows={split[TARGET_COL].notna().sum()}"
    )

# Ensure the date ranges do not overlap.
assert train["DATE"].max() < valid["DATE"].min()
assert valid["DATE"].max() < test["DATE"].min()

train: 1990-05-01 to 2015-09-14 | rows=44689 | target rows=44689
validation: 2015-09-15 to 2021-07-07 | rows=4551 | target rows=4551
test: 2021-07-08 to 2026-09-30 | rows=6773 | target rows=6773


In [5]:
X_train = train[feature_cols]
y_train = train[TARGET_COL]

X_valid = valid[feature_cols]
y_valid = valid[TARGET_COL]

model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("regression", LinearRegression()),
])

model.fit(X_train, y_train)
valid_pred = model.predict(X_valid)

print("Model fitted.")

Model fitted.


In [6]:
def regression_metrics(y_true, y_pred):
    return {
        "MAE_C": mean_absolute_error(y_true, y_pred),
        "RMSE_C": mean_squared_error(y_true, y_pred) ** 0.5,
        "R2": r2_score(y_true, y_pred),
    }

validation_results = regression_metrics(y_valid, valid_pred)
pd.Series(validation_results, name="validation score")

MAE_C     2.448354
RMSE_C    3.095109
R2        0.657782
Name: validation score, dtype: float64

In [7]:
train_valid = pd.concat([train, valid]).sort_values(["DATE", "arrondissement"])

X_train_valid = train_valid[feature_cols]
y_train_valid = train_valid[TARGET_COL]

X_test = test[feature_cols]
y_test = test[TARGET_COL]

final_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("regression", LinearRegression()),
])

final_model.fit(X_train_valid, y_train_valid)
test_pred = final_model.predict(X_test)

test_results = regression_metrics(y_test, test_pred)
pd.Series(test_results, name="test score")

MAE_C     2.392492
RMSE_C    2.996497
R2        0.683219
Name: test score, dtype: float64

In [8]:
# Mean baseline: always predict the mean target from the pre-test data.
mean_baseline = DummyRegressor(strategy="mean")
mean_baseline.fit(X_train_valid, y_train_valid)
mean_pred = mean_baseline.predict(X_test)

comparison = {
    "Linear Regression": test_results,
    "Mean baseline": regression_metrics(y_test, mean_pred),
}

# Persistence baseline: yesterday's target predicts today's target.
target_lag_col = f"{TARGET_COL}_lag1"

if target_lag_col in test.columns:
    persistence_mask = test[target_lag_col].notna()

    if persistence_mask.any():
        persistence_pred = test.loc[persistence_mask, target_lag_col]
        persistence_true = test.loc[persistence_mask, TARGET_COL]

        comparison["Yesterday's temperature"] = regression_metrics(
            persistence_true,
            persistence_pred,
        )

comparison_df = pd.DataFrame(comparison).T
comparison_df

,MAE_C,RMSE_C,R2
Linear Regression,2.392492,2.996497,0.683219
Mean baseline,4.350525,5.547135,-0.085596
Yesterday's temperature,2.476987,3.211317,0.635599
